# Lab Work - 6.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

%matplotlib inline

## Q.1 The Sigmoid Function

### 01 Define the linear score (log-odds): z = w0 + w1x — for a single feature; explain why a raw linear output z is not suitable as a probability output

The linear score z = w₀ + w₁x can range from -∞ to +∞. Probabilities must be between 0 and 1. Hence, we need a function to map z to (0,1).

### 02 Write the sigmoid (logistic) function: σ(z) = 1 / (1 + e^(-z)); compute σ(z) for z ∈ {-5, -2, 0, 2, 5}

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.array([-5, -2, 0, 2, 5])
sigma_values = sigmoid(z_values)

print("z values:", z_values)
print("σ(z) values:", sigma_values)

### 03 Sketch the sigmoid curve... (we'll plot it)

In [ ]:
z = np.linspace(-10, 10, 400)
sigma = sigmoid(z)

plt.figure(figsize=(8, 5))
plt.plot(z, sigma, 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', label='σ(z)=0.5')
plt.axvline(x=0, color='r', linestyle='--')
plt.title('Sigmoid Function')
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.grid(True)
plt.legend()
plt.annotate('Asymptote y=1', xy=(8, 0.95), xytext=(8, 0.85))
plt.annotate('Asymptote y=0', xy=(-8, 0.05), xytext=(-8, 0.15))
plt.show()

### 04 Prove algebraically that σ(z) + σ(-z) = 1

σ(z) = 1 / (1 + e^{-z})
σ(-z) = 1 / (1 + e^{z})

σ(z) + σ(-z) = [e^z + 1] / [(1 + e^z)] = 1

### 05 Derive the derivative: dσ/dz = σ(z)(1 - σ(z))

Let σ(z) = 1 / (1 + e^{-z})

dσ/dz = e^{-z} / (1 + e^{-z})^2 = σ(z) * (1 - σ(z))

### 06 Define the decision rule

ŷ = 1 if σ(z) ≥ 0.5, else 0. This is equivalent to z ≥ 0.

## Q.2 The Decision Boundary

### Dataset: 4 points

In [ ]:
# Dataset
X = np.array([
    [1, 2],
    [2, 3],
    [4, 1],
    [5, 2]
])
y = np.array([0, 0, 1, 1])

print("Features X:", X)
print("Labels y:", y)

### Weights

In [ ]:
w = np.array([-3, 1, 0.5])  # w0, w1, w2

### 02 Compute linear score z = w0 + w1*x1 + w2*x2

In [ ]:
def linear_score(X, w):
    return w[0] + X[:, 0] * w[1] + X[:, 1] * w[2]

z_vals = linear_score(X, w)
print("z values:", z_vals)

### 03 Compute σ(z)

In [ ]:
probs = sigmoid(z_vals)
print("Probabilities:", probs)

### 04 Apply decision rule

In [ ]:
y_pred = (probs >= 0.5).astype(int)
print("Predicted labels:", y_pred)
print("True labels:", y)
print("Misclassified:", np.sum(y_pred != y))

### 05 Decision boundary: w0 + w1 x1 + w2 x2 = 0

In [ ]:
# Solve for x2 = (-w0 - w1 x1) / w2
x1_plot = np.linspace(0, 6, 100)
x2_boundary = (-w[0] - w[1] * x1_plot) / w[2]

plt.figure(figsize=(8, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], color='blue', label='Class 0')
plt.scatter(X[y==1, 0], X[y==1, 1], color='red', label='Class 1')
plt.plot(x1_plot, x2_boundary, 'g--', label='Decision Boundary')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Decision Boundary')
plt.legend()
plt.grid(True)
plt.show()

## Q.3 Binary Cross-Entropy

### 01 Probabilistic model

P(y=1|x) = σ(wᵀx)
P(y=0|x) = 1 - σ(wᵀx)

### 02 Likelihood

L(w) = ∏ P(y_i | x_i)

### 03 Log-likelihood

log L(w) = Σ [y_i log(σ(z_i)) + (1 - y_i) log(1 - σ(z_i))]

### 04 Binary cross-entropy loss

In [ ]:
def binary_cross_entropy(y_true, y_pred_prob):
    return -np.mean(y_true * np.log(y_pred_prob + 1e-15) + (1 - y_true) * np.log(1 - y_pred_prob + 1e-15))

# Compute J(w) for the dataset
J = binary_cross_entropy(y, probs)
print("Cross-entropy loss J(w):", J)

## Q.4 Gradient of the Loss

### Gradient computation

In [ ]:
def compute_gradient(X, y, w):
    m = len(y)
    z = np.dot(np.c_[np.ones((m, 1)), X], w)
    h = sigmoid(z)
    grad = (1/m) * np.dot(np.c_[np.ones((m, 1)), X].T, (h - y))
    return grad

X_bias = np.c_[np.ones((X.shape[0], 1)), X]  # Add bias term
grad = compute_gradient(X, y, w)
print("Gradient ∇J:", grad)

### Gradient descent step

In [ ]:
eta = 0.1
w_new = w - eta * grad
print("Updated weights:", w_new)

### Observation: similarity to Linear Regression gradient

The gradient form (σ(z) - y) * x_j is structurally identical to the gradient of MSE in linear regression due to the exponential family properties.